## 라이브러리 및 상수 선언

In [1]:
# from app_store_scraper import AppStore # 빈데이터를 가져오는 문제로 에러남
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
PACKAGE_LIST = [378084485,1445504255,543831532,1598850912] # 배민, 쿠팡이츠, 요기요, 땡겨요
PACKAGE_NAME = ['baemin', 'coupangeats', 'yogiyo', 'ddangyo']
PACKAGE_NUM = 3
NUM_DATA = 10000

In [3]:
APP_ID = PACKAGE_LIST[PACKAGE_NUM]
APP_NAME = PACKAGE_NAME[PACKAGE_NUM]

## RSS 기반 XML 데이터 불러오기

In [4]:
def fetch_rss_xml(app_id: int, page: int) -> str:
    url = f"https://itunes.apple.com/kr/rss/customerreviews/id={app_id}/page={page}/sortby=mostrecent/xml" # RSS:웹사이트의 새로운 콘텐츠(뉴스, 블로그 글, 공지 등)를 자동으로 받아보는 구독 방식
    r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"}) # "User-Agent": "Mozilla/5.0" : 봇/스크립트가 아닌 브라우저에서 접속한 것처럼 하기 위한 옵션
    r.raise_for_status()
    return r.content.decode("utf-8", errors="replace")

In [5]:
def parse_reviews_from_xml(xml_text: str): # RSS 방식은 XML(태그기반) 데이터이므로 파싱필요.
    root = ET.fromstring(xml_text)

    # 네임스페이스 처리 (url 직접 들어가서 확인 가능)
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "im": "http://itunes.apple.com/rss"
    }

    reviews = []
    entries = root.findall("atom:entry", ns) # 네임스페이스 내 entry 헤더 모두 찾기

    for entry in entries:
        reviewId = entry.find("atom:id",ns).text
        title = entry.find("atom:title", ns).text
        content = entry.find("atom:content", ns).text
        voteSum = int(entry.find("im:voteSum", ns).text) # 도움돼요 점수
        voteCount = int(entry.find("im:voteCount", ns).text) # 전체 투표 수(도움돼요, 도움안돼요)
        score = int(entry.find("im:rating", ns).text)
        at = entry.find("atom:updated", ns).text
        at = pd.to_datetime(at).tz_convert(None) # time-zone 정보 없애고 절대시간으로 변환
        appVersion = entry.find("im:version", ns).text
        userName = entry.find("atom:author/atom:name", ns).text

        reviews.append({
            "reviewId": reviewId,
            "title": title,
            "content": content,
            "voteSum": voteSum,
            "voteCount": voteCount,
            "score": score,
            "at": at,
            "appVersion": appVersion,
            "userName": userName
        })

    return reviews

In [6]:
all_reviews = []
page = 1

while True:
    try:
        xml_text = fetch_rss_xml(APP_ID, page)
        reviews = parse_reviews_from_xml(xml_text)

        # 더 이상 리뷰가 없으면 종료
        if not reviews:
            break

        for r in reviews:
            all_reviews.append(r)
            if len(all_reviews) >= NUM_DATA:
                break

        page += 1
        time.sleep(0.5)

    except Exception as e:
        print(f"[ERROR] app_id={APP_ID}, page={page}, {e}")
        break

[ERROR] app_id=1598850912, page=11, 400 Client Error: Bad Request for url: https://itunes.apple.com/kr/rss/customerreviews/id=1598850912/page=11/sortby=mostrecent/xml


In [7]:
print("수집된 리뷰 수:", len(all_reviews))

수집된 리뷰 수: 500


## 간단한 EDA

In [8]:
df = pd.DataFrame(all_reviews)

In [9]:
df.head()

,reviewId,title,content,voteSum,voteCount,score,at,appVersion,userName
0,13511849538,땡배달 배차도 안잡히는데,이럴거면 땡배달 기능을 왜만듦? \n배차 안잡혀서 30분 뒤에 자동 취소되고 쓸대없...,0,0,1,2025-12-14 07:29:18,1.9.3,ㅈ스스톤
1,13508305881,편해요,매우 편리한앱\n사용성도 대 만족,0,0,5,2025-12-13 10:34:54,1.9.3,으갸갹
2,13507961171,배달을 안 해줍니다,말그대로 배달을 안해주고 배달 완료 처리를 하네요.\n\n판매점에서 기사님이 안 와...,0,0,1,2025-12-13 08:21:24,1.9.3,미르곤
3,13504187145,결제창 오류,ㅅㅂ 비번 두번째 확인할때 왜 백지상태냐!!! 5번째 했는데 왜 안돼냐고!!!! ...,0,0,1,2025-12-12 09:10:54,1.9.3,ㅠ노어터ㅜ난
4,13503241489,ㅋㅋㅋㅋ레전드어플,음식잘못와서 고객센터문의한거 25.10월에 아직도 답변없음 진짜레전드네 고객센터...,0,0,1,2025-12-12 01:41:41,1.9.3,14서버 로제


In [10]:
df["content"] = (
    df["content"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True) # 줄바꿈/탭/중복공백 제거
    .str.strip()
)
df.head()

,reviewId,title,content,voteSum,voteCount,score,at,appVersion,userName
0,13511849538,땡배달 배차도 안잡히는데,이럴거면 땡배달 기능을 왜만듦? 배차 안잡혀서 30분 뒤에 자동 취소되고 쓸대없이 ...,0,0,1,2025-12-14 07:29:18,1.9.3,ㅈ스스톤
1,13508305881,편해요,매우 편리한앱 사용성도 대 만족,0,0,5,2025-12-13 10:34:54,1.9.3,으갸갹
2,13507961171,배달을 안 해줍니다,말그대로 배달을 안해주고 배달 완료 처리를 하네요. 판매점에서 기사님이 안 와 배달...,0,0,1,2025-12-13 08:21:24,1.9.3,미르곤
3,13504187145,결제창 오류,ㅅㅂ 비번 두번째 확인할때 왜 백지상태냐!!! 5번째 했는데 왜 안돼냐고!!!! 짜...,0,0,1,2025-12-12 09:10:54,1.9.3,ㅠ노어터ㅜ난
4,13503241489,ㅋㅋㅋㅋ레전드어플,음식잘못와서 고객센터문의한거 25.10월에 아직도 답변없음 진짜레전드네 고객센터직원...,0,0,1,2025-12-12 01:41:41,1.9.3,14서버 로제


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   reviewId    500 non-null    object        
 1   title       500 non-null    object        
 2   content     500 non-null    object        
 3   voteSum     500 non-null    int64         
 4   voteCount   500 non-null    int64         
 5   score       500 non-null    int64         
 6   at          500 non-null    datetime64[ns]
 7   appVersion  500 non-null    object        
 8   userName    500 non-null    object        
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 35.3+ KB


In [12]:
len(df['reviewId'].unique())

500

In [13]:
df['voteSum'].unique()
df[df['voteSum'] == 2]

,reviewId,title,content,voteSum,voteCount,score,at,appVersion,userName
448,13014352699,땡겨요 페이통장 결제 쿠폰(꼭 읽으셨으면 좋겠네요.),가게 생각하는 앱 좋죠. 배달비 무료도 정말 좋고요. 근데 쿠폰 가지고 소비자 장난...,2,4,1,2025-08-14 10:32:23,1.8.9,쓴다. 내가 리뷰


In [14]:
df['app'] = APP_NAME
df['platform'] = 'appstore'
df.head()

,reviewId,title,content,voteSum,voteCount,score,at,appVersion,userName,app,platform
0,13511849538,땡배달 배차도 안잡히는데,이럴거면 땡배달 기능을 왜만듦? 배차 안잡혀서 30분 뒤에 자동 취소되고 쓸대없이 ...,0,0,1,2025-12-14 07:29:18,1.9.3,ㅈ스스톤,ddangyo,appstore
1,13508305881,편해요,매우 편리한앱 사용성도 대 만족,0,0,5,2025-12-13 10:34:54,1.9.3,으갸갹,ddangyo,appstore
2,13507961171,배달을 안 해줍니다,말그대로 배달을 안해주고 배달 완료 처리를 하네요. 판매점에서 기사님이 안 와 배달...,0,0,1,2025-12-13 08:21:24,1.9.3,미르곤,ddangyo,appstore
3,13504187145,결제창 오류,ㅅㅂ 비번 두번째 확인할때 왜 백지상태냐!!! 5번째 했는데 왜 안돼냐고!!!! 짜...,0,0,1,2025-12-12 09:10:54,1.9.3,ㅠ노어터ㅜ난,ddangyo,appstore
4,13503241489,ㅋㅋㅋㅋ레전드어플,음식잘못와서 고객센터문의한거 25.10월에 아직도 답변없음 진짜레전드네 고객센터직원...,0,0,1,2025-12-12 01:41:41,1.9.3,14서버 로제,ddangyo,appstore


app : 배달앱 이름

platform : 스토어 종류(플레이스토어/앱스토어)

reviewId : id(중복비교를 위해 사용)

title : 리뷰 제목

userName	: 리뷰작성한 사용자 이름

content : 리뷰

score : 별점

voteSum : 투표 "도움돼요" 점수

voteCount : 투표 수

at : 리뷰 작성 날짜

appVersion : 리뷰 수집 당시 앱스토어에 노출되는 현재 앱 버전 정보

저장할 컬럼 [app, platform, reviewId, userName, content, score, voteSum, at]

## CSV 파일 저장

In [15]:
columns = ['app', 'platform', 'reviewId', 'userName', 'content', 'score', 'voteSum', 'at']

In [16]:
df[columns].to_csv(f'{APP_NAME}_reviews_appstore.csv', index=False, encoding="utf-8-sig") # utf-8-sig:윈도우+엑셀에서 한글 깨짐 방지